# 🌬️ Auditoría de datos de la turbina eólica

**Silvana Bamba · Proyecto final de la Unidad 4**  
*CRISP-ML(Q) · Ingeniería de datos*

En este notebook reviso `T1.csv` y ordeno los registros por fecha para
reservar los más recientes para la prueba del modelo. El objetivo del
proyecto es estimar la potencia activa de la turbina en kW.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ruta_csv = Path("datos/T1.csv")
if not ruta_csv.is_file():
    ruta_csv = Path("ciencia-datos/datos/T1.csv")

datos = pd.read_csv(ruta_csv)

print(f"Filas: {len(datos):,} | Columnas: {datos.shape[1]}")
print(f"Duplicados exactos: {datos.duplicated().sum()}")

display(datos.head())
display(pd.DataFrame({
    "Tipo de dato": datos.dtypes.astype(str),
    "Valores faltantes": datos.isna().sum(),
}))

Filas: 50,530 | Columnas: 5
Duplicados exactos: 0


,Date/Time,LV ActivePower (kW),Wind Speed (m/s),Theoretical_Power_Curve (KWh),Wind Direction (°)
0,01 01 2018 00:00,380.047791,5.311336,416.328908,259.994904
1,01 01 2018 00:10,453.769196,5.672167,519.917511,268.641113
2,01 01 2018 00:20,306.376587,5.216037,390.900016,272.564789
3,01 01 2018 00:30,419.645905,5.659674,516.127569,271.258087
4,01 01 2018 00:40,380.650696,5.577941,491.702972,265.674286


,Tipo de dato,Valores faltantes
Date/Time,str,0
LV ActivePower (kW),float64,0
Wind Speed (m/s),float64,0
Theoretical_Power_Curve (KWh),float64,0
Wind Direction (°),float64,0


In [2]:
columna_fecha = "Date/Time"
objetivo = "LV ActivePower (kW)"

datos_ordenados = datos.copy()
datos_ordenados[columna_fecha] = pd.to_datetime(
    datos_ordenados[columna_fecha],
    format="%d %m %Y %H:%M",
    errors="raise",
)

datos_ordenados = datos_ordenados.sort_values(columna_fecha).reset_index(drop=True)
fechas = datos_ordenados[columna_fecha]
intervalos = fechas.diff().dropna()

print(f"Primer registro: {fechas.min()}")
print(f"Último registro: {fechas.max()}")
print(f"Marcas de tiempo repetidas adicionales: {fechas.duplicated().sum()}")
print(f"Intervalos mayores a 10 minutos: {(intervalos > pd.Timedelta(minutes=10)).sum()}")

display(intervalos.value_counts().head().rename("Frecuencia"))

Primer registro: 2018-01-01 00:00:00
Último registro: 2018-12-31 23:50:00
Marcas de tiempo repetidas adicionales: 0
Intervalos mayores a 10 minutos: 32


Date/Time
0 days 00:10:00    50497
0 days 00:20:00       11
0 days 00:40:00        2
0 days 03:00:00        1
0 days 00:50:00        1
Name: Frecuencia, dtype: int64

In [3]:
fecha_corte = fechas.iloc[int(len(datos_ordenados) * 0.80)]

entrenamiento = datos_ordenados.loc[fechas < fecha_corte].copy()
prueba = datos_ordenados.loc[fechas >= fecha_corte].copy()

assert entrenamiento[columna_fecha].max() < prueba[columna_fecha].min()

for nombre, conjunto in [("Entrenamiento", entrenamiento), ("Prueba final", prueba)]:
    periodo = conjunto[columna_fecha]
    print(f"{nombre}: {len(conjunto):,} filas | {periodo.min()} a {periodo.max()}")

Entrenamiento: 40,424 filas | 2018-01-01 00:00:00 a 2018-10-18 22:30:00
Prueba final: 10,106 filas | 2018-10-18 22:40:00 a 2018-12-31 23:50:00


## Fuente del dataset

**Dataset:** [Wind Turbine SCADA Dataset — berkerisen, Kaggle](https://www.kaggle.com/datasets/berkerisen/wind-turbine-scada-dataset/data)

**Archivo utilizado:** `T1.csv`, ubicado en `ciencia-datos/datos/`.

**Variable objetivo:** `LV ActivePower (kW)`.

**Tipo de aprendizaje:** Regresión supervisada, porque se busca estimar
un valor numérico de potencia activa en kilovatios.

## Cierre de la auditoría

Encontré 32 intervalos mayores a 10 minutos. Reservé los 10.106 registros
más recientes para la prueba final, a partir del 18 de octubre de 2018
a las 22:40.